# Heston implied-volatility plots

Recreate the calibrated and non-calibrated IV figures using run 076 from the completed piecewise-constant and Legendre studies. This run has near-median calibration and held-out IV errors across both bases among the 100 saved runs. No calibration is run here.


In [ ]:
from pathlib import Path
import sys
import pickle
import numpy as np

# Works when opened from the dataset folder or from the repository root.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "wiener_chaos/experiments/reporting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the Wiener chaos pricing repository.")
sys.path.insert(0, str(ROOT))
from wiener_chaos.experiments.reporting import (saved_run, save_surface_pair,
                           save_combined_smiles, set_show_figures)

H = ROOT / "Heston"
OUTPUT = H / "Plots"
RUN_ID = 76
PIECEWISE_STUDY = H / "Piecewise constant/Statistics calibration/study"
LEGENDRE_STUDY = H / "Legendre/Statistics calibration/study"
with (H / "heston_data.pkl").open("rb") as f:
    data = pickle.load(f)
piecewise = saved_run(PIECEWISE_STUDY, RUN_ID)
legendre = saved_run(LEGENDRE_STUDY, RUN_ID)
S0 = data["S0"]
set_show_figures(True)  # Show every regenerated figure inline and save its PDF.
print(f"Heston run {RUN_ID:03d}; output: {OUTPUT}")

## Calibrated and non-calibrated implied-volatility figures

Each block writes the 3D IV plot, its absolute-error surface in basis points, and the combined maturity-smile panel. The filenames mirror the paper figure references.

In [ ]:
for section, maturity_key, strike_key, market_key, model_key in [
    ("Calibrated", "maturities_calibr", "strikes_calibr", "iv_calibr", "iv_model"),
    ("Non_calibrated", "maturities_test", "strikes_test", "iv_test", "iv_model_test"),
]:
    folder = OUTPUT / section
    maturities, strikes, market = (data[maturity_key], data[strike_key], data[market_key])
    models = [piecewise[model_key], legendre[model_key]]
    labels = ["Piecewise constant", "Legendre"]
    files, mae_bp = save_surface_pair(maturities, strikes, market, models, labels, S0, folder)
    files.append(save_combined_smiles(maturities, strikes, market, models, labels, S0, folder))
    print(section, "MAE (bp):", {k: round(v, 2) for k, v in mae_bp.items()})
    for path in files:
        print("  saved", path.relative_to(ROOT))